# Aksiya prognoz modellari - Colab ish daftari

Bu notebook Streamlit ilovadagi ayni `model_pipeline.py` kodidan foydalanadi. Shu sababli Colabda modelni yaxshilasangiz, keyin o'sha o'zgarishni loyihaga qaytarib ulash oson bo'ladi.

In [ ]:
!git clone -q https://github.com/Ibroxim1qqq/Induvidual-loyiha.git
%cd Induvidual-loyiha
!pip install -q -r requirements.txt

In [ ]:
import pandas as pd
import plotly.graph_objects as go
import yfinance as yf

from model_pipeline import (
    BASELINE_NAME,
    build_weekly_forecast_bundle,
    evaluate_models,
    evaluate_rolling_horizon_backtests,
    get_horizon_weeks,
    split_train_test,
    summarize_rolling_horizon_backtests,
)

## 1. Data yuklash
Standart sozlama ilovadagi bilan bir xil: 5 yillik tarix va 3 oylik prognoz.

In [ ]:
ticker = 'AAPL'
history_years = 5
forecast_months = 3

raw_data = yf.download(
    ticker,
    period=f'{history_years}y',
    interval='1d',
    auto_adjust=False,
    progress=False,
)
if isinstance(raw_data.columns, pd.MultiIndex):
    raw_data = raw_data.xs(ticker, axis=1, level=1, drop_level=True)
data = raw_data[['Open', 'High', 'Low', 'Close', 'Volume']].dropna().copy()
data.tail()

## 2. Kunlik train-test baholash
Oxirgi 1 yil test sifatida ajratiladi; qolgan tarix train bo'ladi.

In [ ]:
train_data, test_data = split_train_test(data)
metrics_table, predictions, best_model = evaluate_models(data, train_data, test_data)

print('Eng yaxshi kunlik test modeli:', best_model)
metrics_table.round(4)

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=test_data.index, y=test_data['Close'], name='Haqiqiy narx', line=dict(width=3)))
for model_name, series in predictions.items():
    fig.add_trace(go.Scatter(x=series.index, y=series, name=model_name, line=dict(width=1.8)))
fig.update_layout(
    title='Kunlik test davri',
    template='plotly_white',
    hovermode='x unified',
    legend=dict(orientation='h'),
)
fig.show()

## 3. 3 oylik haftalik bashorat va rolling backtest
Bu qism ilovadagi asosiy kelajak prognozi bilan bir xil pipeline'dan foydalanadi.

In [ ]:
horizon_weeks = get_horizon_weeks(forecast_months)
future_forecasts = build_weekly_forecast_bundle(data, horizon_weeks)
rolling_details = evaluate_rolling_horizon_backtests(data, forecast_months)
rolling_summary = summarize_rolling_horizon_backtests(rolling_details)

rolling_summary.round(4)

In [ ]:
weekly_history = data['Close'].resample('W-FRI').last().dropna().tail(52)
fig = go.Figure()
fig.add_trace(go.Scatter(x=weekly_history.index, y=weekly_history, name='Tarixiy narx', line=dict(width=3)))
for model_name in future_forecasts.columns:
    fig.add_trace(go.Scatter(x=future_forecasts.index, y=future_forecasts[model_name], name=model_name))
fig.update_layout(
    title='3 oylik kelajak bashorati',
    template='plotly_white',
    hovermode='x unified',
    legend=dict(orientation='h'),
)
fig.show()

## 4. Qanday yaxshilash kerak

- Yangi model qo'shish yoki parametrni o'zgartirish uchun `model_pipeline.py` faylini tahrir qiling.
- Colabda natijani tekshiring.
- Yaxshi natija chiqqach, o'sha o'zgarishni GitHubga push qiling; Streamlit ilova ham ayni kodni ishlatadi.